# Sentiment Analysis — Synthetic vs. Real Evaluation

Evaluates the GET pipeline's corruption strategy on `cardiffnlp/tweet_eval` (sentiment split).
- **Generation**: DeepSeek rewrites tweets using 6 error types (flip-negative, flip-positive, sarcasm, negation, intensity-reduction, paraphrase)
- **Models**: BERTweet and multilingual DistilBERT, both fine-tuned on TweetEval sentiment
- **Hypothesis**: High model accuracy on synthetic data → the corruption is realistic enough to stress-test the classifiers

## 1 · Setup & Load

In [ ]:
import sys, os, json, glob
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

RUNS_DIR = os.path.join(ROOT, "framework", "data", "runs", "sentiment_corruption")
sessions = sorted(glob.glob(os.path.join(RUNS_DIR, "*")), reverse=True)
assert sessions, f"No sessions found in {RUNS_DIR} — run the pipeline first."

SESSION = sessions[0]
print(f"Session: {os.path.basename(SESSION)}")

# ── Load results.json ────────────────────────────────────────────────────────
with open(os.path.join(SESSION, "results.json"), encoding="utf-8") as f:
    pipeline = json.load(f)
meta    = pipeline["meta"]
results = pipeline["results"]
print(f"Runs completed: {meta['runs_completed']} / {meta['num_runs']}")
if meta.get("partial"):
    print("WARNING: pipeline still running — results are partial.")

# ── Load generated data (all runs) ──────────────────────────────────────────
run_files = sorted(glob.glob(os.path.join(SESSION, "generated", "run_*.json")))
generated = []
for p in run_files:
    with open(p, encoding="utf-8") as f:
        generated.extend(json.load(f))
print(f"Generated samples: {len(generated)} across {len(run_files)} run(s)")

# ── Load real sample ─────────────────────────────────────────────────────────
real_sample_path = os.path.join(SESSION, "real_sample.json")
if os.path.exists(real_sample_path):
    with open(real_sample_path, encoding="utf-8") as f:
        real = json.load(f)  # already {"text", "label"} format
    print(f"Real samples: {len(real)}")
else:
    real = []
    print("No real_sample.json found — real baseline charts skipped.")

# ── Constants ────────────────────────────────────────────────────────────────
BERTWEET = "finiteautomata/bertweet-base-sentiment-analysis"
MULTI    = "lxyuan/distilbert-base-multilingual-cased-sentiments-student"
MODELS   = {"BERTweet": BERTWEET, "Multilingual": MULTI}
COLORS   = {"BERTweet": "#3498db", "Multilingual": "#e67e22"}
METRICS  = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
ET_ORDER = ["sentiment_flip_negative", "sentiment_flip_positive", "sarcasm_injection",
            "negation_insertion", "intensity_reduction", "paraphrase"]

## 2 · Metrics Overview

In [ ]:
def get_score(results, model_key, split, metric):
    val = results.get(model_key, {}).get(split, {}).get(metric, {})
    if isinstance(val, dict):
        return val.get("mean", 0.0), val.get("std", 0.0)
    return float(val) if val else 0.0, 0.0

# Summary table
rows = []
for label, key in MODELS.items():
    for m in METRICS:
        mean_g, std_g = get_score(results, key, "generated", m)
        mean_r, std_r = get_score(results, key, "real",      m)
        rows.append({"model": label, "metric": m,
                     "generated": f"{mean_g:.3f} ± {std_g:.3f}",
                     "real": f"{mean_r:.3f}" if mean_r else "—"})
print(pd.DataFrame(rows).to_string(index=False))

# ── Chart: generated vs real per metric ─────────────────────────────────────
fig, axes = plt.subplots(1, len(METRICS), figsize=(14, 4), sharey=True)
for ax, metric in zip(axes, METRICS):
    for i, (label, key) in enumerate(MODELS.items()):
        color = COLORS[label]
        mean_g, std_g = get_score(results, key, "generated", metric)
        mean_r, std_r = get_score(results, key, "real",      metric)
        # two separate bars: generated (solid) and real (hatched)
        ax.bar(i - 0.2, mean_g, 0.35, color=color, alpha=1.0,
               yerr=std_g, capsize=4, error_kw={"elinewidth": 1})
        ax.bar(i + 0.2, mean_r, 0.35, color=color, alpha=0.4,
               yerr=std_r, capsize=4, error_kw={"elinewidth": 1})
    ax.set_title(metric.replace("_", "\n"), fontsize=9)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(list(MODELS.keys()), fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.set_axisbelow(True)

legend_handles = []
for label, c in COLORS.items():
    legend_handles += [
        Patch(color=c, alpha=1.0,  label=f"{label} (generated)"),
        Patch(color=c, alpha=0.4,  label=f"{label} (real)"),
    ]
fig.legend(handles=legend_handles, loc="upper right", fontsize=8, ncol=2)
fig.suptitle("Sentiment Models — Generated vs. Real Benchmark", fontsize=11)
fig.tight_layout()
plt.savefig("sentiment_overview.png", dpi=150)
plt.show()

## 3 · Error Type Distribution

In [ ]:
et_counts = Counter(s.get("error_type", "unknown") for s in generated)
counts    = [et_counts.get(et, 0) for et in ET_ORDER]
total     = sum(counts)

print("Error type distribution:")
for et, c in zip(ET_ORDER, counts):
    print(f"  {et:<35} {c:>4}  ({100*c/max(total,1):.1f}%)")

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(ET_ORDER)), counts, color="#9b59b6", edgecolor="white")
ax.bar_label(bars, fmt="%d", padding=3, fontsize=8)
ax.set_ylabel("Count")
ax.set_title("Synthetic Samples per Error Type")
ax.set_xticks(range(len(ET_ORDER)))
ax.set_xticklabels(ET_ORDER, rotation=25, ha="right", fontsize=9)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
fig.tight_layout()
plt.savefig("sentiment_error_dist.png", dpi=150)
plt.show()

## 4 · Per-Error-Type Accuracy

Which corruption types are hardest for each model?
(`paraphrase` is excluded — no deterministic ground-truth label.)

In [ ]:
from framework.tasks.sentiment.task import SentimentTask
from framework.models.sentiment.transformer import TransformerSentimentModel

TASK = SentimentTask()

eval_items = []
for s in generated:
    label = TASK.get_label(s)
    if label is None:
        continue
    eval_items.append({"text": s["corrupted"], "label": label, "error_type": s["error_type"]})
print(f"Eval items (excl. paraphrase): {len(eval_items)}")

print("Loading models...")
model_objs = {
    "BERTweet": TransformerSentimentModel({
        "name": BERTWEET, "type": "bertweet", "max_length": 128,
        "label_map": {"POS": "POSITIVE", "NEU": "NEUTRAL", "NEG": "NEGATIVE"}
    }),
    "Multilingual": TransformerSentimentModel({
        "name": MULTI, "type": "multilingual", "max_length": 256,
        "label_map": {"positive": "POSITIVE", "neutral": "NEUTRAL", "negative": "NEGATIVE"}
    }),
}

texts = [e["text"] for e in eval_items]
preds = {name: m.predict(texts) for name, m in model_objs.items()}

et_acc = {name: {} for name in MODELS}
for name in MODELS:
    for et in ET_ORDER:
        indices = [i for i, e in enumerate(eval_items) if e["error_type"] == et]
        if not indices:
            et_acc[name][et] = None
            continue
        correct = sum(preds[name][i] == eval_items[i]["label"] for i in indices)
        et_acc[name][et] = correct / len(indices)

df_et = pd.DataFrame(et_acc, index=ET_ORDER).round(3)
print(df_et.to_string())

In [ ]:
x = np.arange(len(ET_ORDER))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, color) in enumerate(COLORS.items()):
    vals  = [et_acc[name].get(et) for et in ET_ORDER]
    xs    = [x[j] + (i - 0.5) * width for j, v in enumerate(vals) if v is not None]
    vs    = [v for v in vals if v is not None]
    if vs:
        ax.bar(xs, vs, width, label=name, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(ET_ORDER, rotation=25, ha="right", fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Accuracy")
ax.set_title("Per-Error-Type Accuracy")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="chance (0.5)")
ax.legend()
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
fig.tight_layout()
plt.savefig("sentiment_per_error_acc.png", dpi=150)
plt.show()

## 5 · Label Distribution — Real vs. Synthetic

In [ ]:
LABELS = ["NEGATIVE", "NEUTRAL", "POSITIVE"]

real_counts  = Counter(r["label"] for r in real)
synth_counts = Counter(TASK.get_label(s) for s in generated if TASK.get_label(s) is not None)

real_p  = [real_counts.get(l, 0)  / max(sum(real_counts.values()),  1) for l in LABELS]
synth_p = [synth_counts.get(l, 0) / max(sum(synth_counts.values()), 1) for l in LABELS]

print(f"{'Label':<12} {'Real':>8} {'Synthetic':>12}")
for l, r, s in zip(LABELS, real_p, synth_p):
    print(f"{l:<12} {r:>8.3f} {s:>12.3f}")

x = np.arange(len(LABELS))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, real_p,  0.35, label="Real",      color="#3498db")
ax.bar(x + 0.2, synth_p, 0.35, label="Synthetic", color="#e67e22", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_ylabel("Proportion")
ax.set_title("Label Distribution — Real vs. Synthetic")
ax.legend()
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
fig.tight_layout()
plt.savefig("sentiment_label_dist.png", dpi=150)
plt.show()

## 6 · Text Length Distribution

In [ ]:
real_len  = [len(r["text"].split())      for r in real]
synth_len = [len(s["corrupted"].split()) for s in generated]

if real_len:
    print(f"Real      — mean {np.mean(real_len):.1f} words, median {np.median(real_len):.1f}")
print(f"Synthetic — mean {np.mean(synth_len):.1f} words, median {np.median(synth_len):.1f}")

fig, ax = plt.subplots(figsize=(9, 4))
all_lens = real_len + synth_len
bins = range(0, max(all_lens) + 6, 5) if all_lens else range(0, 50, 5)
if real_len:
    ax.hist(real_len,  bins=bins, alpha=0.6, label="Real",      color="#3498db")
ax.hist(synth_len, bins=bins, alpha=0.6, label="Synthetic", color="#e67e22")
ax.set_xlabel("Word count")
ax.set_ylabel("Frequency")
ax.set_title("Text Length Distribution — Real vs. Synthetic")
ax.legend()
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_axisbelow(True)
fig.tight_layout()
plt.savefig("sentiment_length_dist.png", dpi=150)
plt.show()

## 7 · Sample Examples per Error Type

In [ ]:
by_type = {et: [] for et in ET_ORDER}
for s in generated:
    et = s.get("error_type")
    if et in by_type:
        by_type[et].append(s)

for et in ET_ORDER:
    items = by_type[et][:2]
    if not items:
        continue
    label = TASK.get_label({"error_type": et})
    print(f"\n{'─'*70}")
    print(f"[{et}]  → ground truth: {label}")
    for s in items:
        print(f"  original : {s.get('original','')[:110]}")
        print(f"  corrupted: {s.get('corrupted','')[:110]}")
        print()